# Lyapunov Safety-Filter MPC Target-Selector Term Ablation

This notebook mirrors `LyapunovSafetyFilterMPC.ipynb`, but it sweeps exact target-selector objective-term activations to compare which selector terms matter most in the closed loop. Each study uses the same safety-MPC backend and debug/export flow as the current safety MPC notebook, except fallback MPC is intentionally disabled; only the selector objective mask changes.


In [ ]:
import copy
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from Simulation.mpc import MpcSolver, compute_observer_gain
from Simulation.run_mpc_lyapunov import run_mpc_lyapunov
from Simulation.system_functions import PolymerCSTR
from Lyapunov.safety_debug import build_safety_filter_run_bundle, save_safety_filter_debug_artifacts
from utils.scaling_helpers import apply_min_max, reverse_min_max
from utils.td3_helpers import load_and_prepare_system_data


In [ ]:
# Polymer CSTR setup
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()
# Match the original MPC notebook convention: the plant object stays in physical coordinates.
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_phys = np.array([[4.5, 324.0], [3.4, 321.0]])

system_dict_path = os.path.join("Data", "system_dict")
augmentation_style = "rawlings"
augmentation_mode = "mixed_B_I"

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_phys,
    u_min=u_min,
    u_max=u_max,
    system_dict_path=system_dict_path,
    augmentation_style=augmentation_style,
    augmentation_mode=augmentation_mode,
)

A = system_data["A"]
B = system_data["B"]
C = system_data["C"]
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
Bd_used = system_data["Bd_used"]
Cd_used = system_data["Cd_used"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

n_tests = 2
set_points_len = 400
TEST_CYCLE = [False, False]
warm_start = 0

poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                  0.42900673, 0.4228262, 0.96916776, 0.91230187])
L = compute_observer_gain(A_aug, C_aug, poles)

In [ ]:
# MPC configuration and reward
inputs_number = int(B_aug.shape[1])
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
b_min = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
b_max = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])
b1 = (b_min[0] - u_ss[0], b_max[0] - u_ss[0])
b2 = (b_min[1] - u_ss[1], b_max[1] - u_ss[1])
bnds = (b1, b2) * cont_h
cons = []
IC_opt = np.zeros(inputs_number * cont_h)

MPC_obj = MpcSolver(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([5.0, 1.0]),
    R_in=np.array([1.0, 1.0]),
    NP=predict_h,
    NC=cont_h,
)

reward_config, reward_fn = make_reward_fn_mpc_quadratic(
    Q_diag=np.array([5.0, 1.0]),
    R_diag=np.array([1.0, 1.0]),
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

Qs_tgt_diag = 1e8 * np.asarray(MPC_obj.Q_out, float)
Ru_tgt_diag = 1.0 * np.ones(MPC_obj.B.shape[1])

In [ ]:
# Run MPC upstream + Lyapunov safety filter selector-term ablations
selector_config = {
    "alpha_u_ref": 0.5,
    "alpha_du_sel": 0.5,
    "alpha_dx_sel": 0.05,
    "alpha_x_ref": 0.01,
    "x_weight_base": "CtQC",
    "use_output_bounds_in_selector": True,
}

run_config = {
    "rho_lyap": 0.98,
    "lyap_eps": 1e-9,
    "lyap_tol": 1e-10,
    "w_mpc": 1.0,
    "w_track": 1.0,
    "w_move": 0.2,
    "w_ss": 0.0,
    "fallback_policy": None,
    "mpc_target_policy": "raw_setpoint",
    "tracking_target_policy": "raw_setpoint",
    "target_selector_config": selector_config,
    "selector_H": None,
    "target_backup_policy": "last_valid",
    "selector_warm_start": True,
    "lyap_acceptance_mode": "hard_only",
    "reuse_mpc_solution_as_ic": False,
    "reset_system_on_entry": True,
    "allow_lyap_slack": True,
    "trust_region_delta": 0.15,
    "allow_trust_region_slack": False,
}
print("Using refined Step A selector")
print("Fallback MPC disabled for selector-term ablation runs")

SELECTOR_TERMS = (
    "target_tracking",
    "u_applied_anchor",
    "u_prev_smoothing",
    "x_prev_smoothing",
    "xhat_anchor",
)

term_studies = [
    {"study_name": "all_terms_on", "active_terms": list(SELECTOR_TERMS)},
    {"study_name": "objective_zero", "active_terms": []},
    {"study_name": "only_target_tracking", "active_terms": ["target_tracking"]},
    {"study_name": "only_u_applied_anchor", "active_terms": ["u_applied_anchor"]},
    {"study_name": "only_u_prev_smoothing", "active_terms": ["u_prev_smoothing"]},
    {"study_name": "only_x_prev_smoothing", "active_terms": ["x_prev_smoothing"]},
    {"study_name": "only_xhat_anchor", "active_terms": ["xhat_anchor"]},
]


def make_term_activation(active_terms):
    active_terms = set(active_terms)
    return {term: (term in active_terms) for term in SELECTOR_TERMS}


def safe_nanmean(arr):
    arr = np.asarray(arr, float)
    return float(np.nanmean(arr)) if arr.size and np.any(np.isfinite(arr)) else np.nan


def to_physical_setpoint(bundle):
    n_u = bundle["u_applied_phys"].shape[1]
    y_ss_scaled = apply_min_max(steady_states["y_ss"], data_min[n_u:], data_max[n_u:])
    return reverse_min_max(bundle["y_sp"] + y_ss_scaled, data_min[n_u:], data_max[n_u:])


def output_rmse_post_step(bundle):
    y_sp_phys = to_physical_setpoint(bundle)
    y_post = np.asarray(bundle["y_system"][1 : 1 + y_sp_phys.shape[0], :], float)
    return np.sqrt(np.mean((y_post - y_sp_phys[: y_post.shape[0], :]) ** 2, axis=0))


study_root = os.path.join(
    os.getcwd(),
    "Data",
    "debug_exports",
    "mpc_selector_term_ablation",
    datetime.now().strftime("%Y%m%d_%H%M%S"),
)
os.makedirs(study_root, exist_ok=True)

comparison_records = []
study_debug_dirs = {}

for study in term_studies:
    study_name = study["study_name"]
    active_terms = list(study["active_terms"])
    term_activation = make_term_activation(active_terms)

    study_run_config = copy.deepcopy(run_config)
    study_run_config["target_selector_config"] = copy.deepcopy(run_config["target_selector_config"])
    study_run_config["target_selector_config"]["term_activation"] = term_activation

    cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)

    results = run_mpc_lyapunov(
        system=cstr,
        MPC_obj=MPC_obj,
        y_sp_scenario=y_sp_scenario,
        n_tests=n_tests,
        set_points_len=set_points_len,
        steady_states=steady_states,
        IC_opt=IC_opt,
        bnds=bnds,
        cons=cons,
        warm_start=warm_start,
        L=L,
        data_min=data_min,
        data_max=data_max,
        test_cycle=TEST_CYCLE,
        reward_fn=reward_fn,
        nominal_qi=nominal_qi,
        nominal_qs=nominal_qs,
        nominal_ha=nominal_hA,
        qi_change=qi_change,
        qs_change=qs_change,
        ha_change=ha_change,
        mode="disturb",
        use_lyap=True,
        rho_lyap=study_run_config["rho_lyap"],
        lyap_eps=study_run_config["lyap_eps"],
        lyap_tol=study_run_config["lyap_tol"],
        w_mpc=study_run_config["w_mpc"],
        w_track=study_run_config["w_track"],
        w_move=study_run_config["w_move"],
        w_ss=study_run_config["w_ss"],
        Qy_track_diag=None,
        Rmove_diag=None,
        Qs_tgt_diag=Qs_tgt_diag,
        Ru_tgt_diag=Ru_tgt_diag,
        allow_lyap_slack=study_run_config["allow_lyap_slack"],
        trust_region_delta=study_run_config["trust_region_delta"],
        allow_trust_region_slack=study_run_config["allow_trust_region_slack"],
        fallback_policy=study_run_config["fallback_policy"],
        target_selector_config=study_run_config["target_selector_config"],
        selector_H=study_run_config["selector_H"],
        mpc_target_policy=study_run_config["mpc_target_policy"],
        tracking_target_policy=study_run_config["tracking_target_policy"],
        target_backup_policy=study_run_config["target_backup_policy"],
        selector_warm_start=study_run_config["selector_warm_start"],
        lyap_acceptance_mode=study_run_config["lyap_acceptance_mode"],
        reuse_mpc_solution_as_ic=study_run_config["reuse_mpc_solution_as_ic"],
        reset_system_on_entry=study_run_config["reset_system_on_entry"],
    )

    bundle = build_safety_filter_run_bundle(
        source="mpc",
        results=results,
        steady_states=steady_states,
        config=study_run_config,
        min_max_dict=min_max_dict,
        data_min=data_min,
        data_max=data_max,
        extra={"reward_config": reward_config},
    )

    debug_dir = save_safety_filter_debug_artifacts(
        bundle=bundle,
        directory=study_root,
        prefix_name=study_name,
        save_plots=True,
    )
    study_debug_dirs[study_name] = debug_dir

    rmse = output_rmse_post_step(bundle)
    comparison_records.append(
        {
            "study_name": study_name,
            "active_terms": ", ".join(active_terms) if active_terms else "none",
            "reward_mean": bundle["summary"].get("reward_mean"),
            "target_error_inf_mean": safe_nanmean(bundle["target_error_inf"]),
            "target_error_inf_max": bundle["summary"].get("target_error_inf_max"),
            "n_target_success": bundle["summary"].get("n_target_success"),
            "n_qcqp_attempted": bundle["summary"].get("n_qcqp_attempted"),
            "n_qcqp_hard_accepted": bundle["summary"].get("n_qcqp_hard_accepted"),
            "output0_rmse": float(rmse[0]),
            "output1_rmse": float(rmse[1]),
            "mode_counts": str(bundle["summary"].get("mode_counts")),
            "debug_dir": debug_dir,
        }
    )
    print(f"Completed {study_name}: {debug_dir}")

comparison_df = pd.DataFrame(comparison_records)
comparison_df = comparison_df.sort_values(by="reward_mean", ascending=False, na_position="last").reset_index(drop=True)
comparison_csv = os.path.join(study_root, "comparison_table.csv")
comparison_df.to_csv(comparison_csv, index=False)

plt.figure(figsize=(11, 4.5))
plt.bar(comparison_df["study_name"], comparison_df["reward_mean"], color="#2A6F97")
plt.xticks(rotation=30, ha="right")
plt.ylabel("reward_mean")
plt.title("Target-Selector Term Ablation: Reward Mean")
plt.tight_layout()
reward_plot_path = os.path.join(study_root, "comparison_reward_mean.png")
plt.savefig(reward_plot_path, dpi=300, bbox_inches="tight")
plt.show()

x = np.arange(len(comparison_df))
width = 0.38
plt.figure(figsize=(11, 4.5))
plt.bar(x - width / 2, comparison_df["output0_rmse"], width=width, label="output0_rmse", color="#BC4749")
plt.bar(x + width / 2, comparison_df["output1_rmse"], width=width, label="output1_rmse", color="#386641")
plt.xticks(x, comparison_df["study_name"], rotation=30, ha="right")
plt.ylabel("RMSE (physical units)")
plt.title("Target-Selector Term Ablation: Output RMSE")
plt.legend()
plt.tight_layout()
rmse_plot_path = os.path.join(study_root, "comparison_output_rmse.png")
plt.savefig(rmse_plot_path, dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 4.5))
plt.bar(comparison_df["study_name"], comparison_df["target_error_inf_max"], color="#7B2CBF")
plt.xticks(rotation=30, ha="right")
plt.ylabel("target_error_inf_max")
plt.title("Target-Selector Term Ablation: Target Error Infinity Norm")
plt.tight_layout()
target_plot_path = os.path.join(study_root, "comparison_target_error_inf_max.png")
plt.savefig(target_plot_path, dpi=300, bbox_inches="tight")
plt.show()

display(comparison_df)
print("Study root:", study_root)
print("Comparison CSV:", comparison_csv)
print("Reward plot:", reward_plot_path)
print("RMSE plot:", rmse_plot_path)
print("Target-error plot:", target_plot_path)
